# Shakespeare task

In [7]:
# Import required libraries
import numpy as np
import requests

# URL of the Shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

# Fetching the text data from the URL
response = requests.get(url)
text = response.text

# Printing the length of the fetched text
print("Length of text: ", len(text))

Length of text:  1115394


In [ ]:
# Alternativly you may use the input.txt file provided in your folder

### Step 1: Prepare Text Data, Character Mappings, and Sequences
We will extract the unique characters (vocabulary), create mappings from characters to indices and vice versa, and slice the text into semi-overlapping sequences of a fixed length (`maxlen`).

In [8]:
# Get unique characters and create mapping dictionaries
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_indices = {char: i for i, char in enumerate(chars)}
indices_char = {i: char for i, char in enumerate(chars)}

print(f"Unique characters (vocab size): {vocab_size}")

# Cut the text in semi-redundant sequences of maxlen characters
maxlen = 40
step = 3
sentences = []
next_chars = []

# We will use a subset of the text to keep training fast and avoid running out of memory
subset_text = text[:100000]

for i in range(0, len(subset_text) - maxlen, step):
    sentences.append(subset_text[i : i + maxlen])
    next_chars.append(subset_text[i + maxlen])

print(f"Number of sequences: {len(sentences)}")

Unique characters (vocab size): 65
Number of sequences: 33320


### Step 2: Vectorization
We will convert the text sequences into binary matrices (one-hot encoding) suitable for training our network.

In [9]:
import numpy as np

# Vectorize input sequences and targets
x = np.zeros((len(sentences), maxlen, vocab_size), dtype=bool)
y = np.zeros((len(sentences), vocab_size), dtype=bool)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        x[i, t, char_indices[char]] = 1
    y[i, char_indices[next_chars[i]]] = 1

print("Input shape (x):", x.shape)
print("Target shape (y):", y.shape)

Input shape (x): (33320, 40, 65)
Target shape (y): (33320, 65)


### Step 3: Build and Train the Simple RNN Model
We will import the Keras library and build a SimpleRNN architecture, then train it for 10 epochs.

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from tensorflow.keras.optimizers import RMSprop

# Build a Simple RNN Model
model = Sequential([
    SimpleRNN(128, input_shape=(maxlen, vocab_size)),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer=RMSprop(learning_rate=0.01))
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 65)             │         8,385 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,217 (129.75 KB)

 Trainable params: 33,217 (129.75 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Train the model for 10 epochs
epochs = 10
batch_size = 128

model.fit(x, y, batch_size=batch_size, epochs=epochs)

Epoch 1/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 12s 41ms/step - loss: 3.3661
Epoch 2/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 42ms/step - loss: 2.7256
Epoch 3/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - loss: 2.5101
Epoch 4/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 40ms/step - loss: 2.5023
Epoch 5/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 41ms/step - loss: 2.4659
Epoch 6/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 19s 35ms/step - loss: 2.4450
Epoch 7/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 41ms/step - loss: 2.5594
Epoch 8/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 43ms/step - loss: 2.6048
Epoch 9/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 40ms/step - loss: 2.5440
Epoch 10/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - loss: 2.7471


### Step 4: Text Generation Function
This helper function samples the next character probabilistically (using temperature to control diversity/coherence) and runs until exactly 100 words are generated.

In [5]:
def sample(preds, temperature=1.0):
    # Helper function to sample an index from a probability array
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-10) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

def generate_text(model, seed_text, word_count=100, temperature=0.5):
    # Clean the seed text or pad it to match maxlen
    generated = seed_text

    while True:
        # Ensure we only check the word count of the newly generated additions + seed
        words = generated.split()
        if len(words) >= word_count:
            # Truncate precisely to the specified word count
            return " ".join(words[:word_count])

        # Prepare the input sequence from the end of current text
        input_sequence = generated[-maxlen:]
        # Pad with spaces if the text is shorter than maxlen
        if len(input_sequence) < maxlen:
            input_sequence = input_sequence.rjust(maxlen)

        # Vectorize the sequence
        x_pred = np.zeros((1, maxlen, vocab_size))
        for t, char in enumerate(input_sequence):
            if char in char_indices:
                x_pred[0, t, char_indices[char]] = 1.0

        # Predict probabilities
        preds = model.predict(x_pred, verbose=0)[0]
        next_index = sample(preds, temperature)
        next_char = indices_char[next_index]

        generated += next_char

### Step 5: Generate and Print Sample Text
Let's test our text generation function with a seed text and print the 100-word output.

In [12]:
# Define seed text
seed_text = "First Citizen:\nBefore we proceed any further, hear me speak.\n"

# Generate sample text
generated_sample = generate_text(model, seed_text, word_count=100, temperature=0.6)

print("--- Generated Text (100 words) ---\n")
print(generated_sample)
print("\n---------------------------------")
print("Word count verify:", len(generated_sample.split()))

--- Generated Text (100 words) ---

First Citizen: Before we proceed any further, hear me speak. IUSUS: he de he deece ded le whe den wos he bhe shand hes he hond hherm Br he hes he he ded des hed had hed ded eeshe denis medeurs he d d de he d bhe he he dhendond mid des wherd de hed he haces th oud he he d phane thend the y is ant hes en fhed d hended hes hh he he ded had hin the the mhe hhh dell bas he de hh n he er hhe d des deiners dedend'te sirhe h

---------------------------------
Word count verify: 100
